In [323]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from collections import Counter
import math

# Theme of the Project

This project investigates the mathematical foundations of cryptography through the implementation and analysis of classical and modern encryption schemes. Using Python and NumPy, we explore modular arithmetic and linear algebra as the theoretical basis for the Caesar cipher, Vigenère cipher, and the matrix-based Hill cipher, followed by a simplified demonstration of RSA encryption and cryptographic hash functions. Each notebook focuses on the underlying mathematics of one cipher, bridging the gap between abstract mathematical concepts and their practical application in secure communication.

## Table of Contents

1. Mathematical Foundations
   - 1.1 Modular Arithmetic
   - 1.2 GCD & The Euclidean Algorithm
   - 1.3 Modular Inverse
   - 1.4 Matrix Operations in Cryptography
2. Classical Ciphers
   - 2.1 Caesar Cipher
   - 2.2 Vigenère Cipher
   - 2.3 Frequency Analysis
3. Hill Cipher
   - 3.1 Encryption
   - 3.2 Decryption

4. Modern Cryptography
   - 4.1 RSA
   - 4.2 Hashing
   - 4.3 AES — Symmetric Encryption
   - 4.4 Cryptographically Secure Randomness
   - 4.5 The Quantum Computing Threat

5. Conclusion

## 1. Mathematical Foundations

Before implementing any cipher, we establish the three mathematical 
tools that all cryptographic schemes in this project rely on:

- **Modular arithmetic** — the algebra of remainders, forming the 
  basis of all letter transformations
- **Greatest Common Divisor (GCD)** — used to determine whether a 
  number or matrix is invertible in modular space
- **Matrix operations (mod n)** — the foundation of block ciphers 
  such as the Hill cipher, where encryption is a linear transformation
  over a finite field

### 1.1 Modular Arithmetic

Modular arithmetic is a mathematical system where integers "wrap around" 
after exceeding a given value called the **modulus**. Formally:

$$a \equiv b \pmod{m} \iff m \mid (a - b)$$

We say $a$ is congruent to $b$ modulo $m$ if $m$ divides their difference.

For example:

$$3 \equiv 15 \pmod{12} \quad \text{since } 12 \mid (15 - 3)$$
$$70 \equiv 10 \pmod{60} \quad \text{since } 60 \mid (70 - 10)$$

In cryptography, modular arithmetic keeps all values within a fixed range.
For classical ciphers this range is $\mathbb{Z}_{26} = \{0, 1, \dots, 25\}$,
where each integer represents a letter of the English alphabet:

$$27 \equiv 1 \pmod{26}$$
$$34 \equiv 8 \pmod{26}$$

The following cell shows the implementation in python:

In [2]:
print(27 % 26)  # 1
print(34 % 26)  # 8
print(-6 % 26)

1
8
20



### 1.2 Greatest Common Divisor (GCD)

The Greatest Common Divisor of two integers is the largest integer 
that divides both without a remainder:

$$\gcd(a, b) = \max\{d \in \mathbb{Z}^+ : d \mid a \text{ and } d \mid b\}$$

Two integers are called **coprime** if their GCD equals 1:

$$\gcd(a, b) = 1$$

This is not an edge case — it is the fundamental requirement in 
cryptography. A number $a$ has a modular inverse mod $m$ **if and 
only if** $\gcd(a, m) = 1$.

### The Euclidean Algorithm

Rather than factoring both numbers, we use the recursive observation:

$$\gcd(a, b) = \gcd(b,\ a \bmod b)$$

repeated until the remainder reaches zero. For example:

$$\gcd(48, 18) \rightarrow \gcd(18, 12) \rightarrow \gcd(12, 6) 
\rightarrow \gcd(6, 0) = 6$$

In the context of this project, the condition we require is:

$$\gcd(a, 26) = 1$$

For example $\gcd(3, 26) = 1$ so $3$ is a valid key, 
while $\gcd(13, 26) = 13$ so $13$ is not.

The following cell shows the implementation in python:

In [3]:
def findGCD(a,b):
    if b == 0:
        return a
    return findGCD(b,a % b)    

In [4]:
print(findGCD(3,26))
print(findGCD(18,48))
print(findGCD(2,4))


1
6
2


SymPy provides a built-in gcd function which we use to verify our implementation:

In [5]:
assert sp.gcd(3,26) == findGCD(3,26)
assert sp.gcd(18,48) == findGCD(18,48)
assert sp.gcd(2,4) == findGCD(2,4)
assert sp.gcd(47,123) == findGCD(47,123) 


### 1.3 Modular Inverse

The modular inverse of $a$ modulo $m$ is an integer $x$ such that:

$$a \cdot x \equiv 1 \pmod{m}$$

For example, the modular inverse of $3$ mod $26$ is $9$, since:

$$3 \times 9 = 27 \equiv 1 \pmod{26}$$

A modular inverse exists **if and only if**:

$$\gcd(a, m) = 1$$

We find it using the **Extended Euclidean Algorithm**, which finds 
integers $x$ and $y$ satisfying Bézout's identity:

$$ax + my = \gcd(a, m)$$

When $\gcd(a, m) = 1$, the coefficient $x$ reduced mod $m$ is 
the modular inverse of $a$.

The following cell shows the implementation in python: 

In [6]:
def extended_euclidian(a,modular):
    curr_number = a
    next_number = modular
    current_coefficient = 1
    next_coefficient = 0
    while next_number != 0 : 
        q = curr_number // next_number
        curr_number , next_number = next_number, curr_number - next_number * q
        current_coefficient , next_coefficient = next_coefficient , current_coefficient - next_coefficient * q
    return curr_number,current_coefficient 
def mod_inverse(a,modular):
    gcd , x = extended_euclidian(a,modular)
    if gcd != 1:
        return None
    return x % modular         

In [7]:
print(mod_inverse(3 , 26))

9


SymPy provides a built-in mod_inverse function which we use to verify our implementation:

In [8]:
assert sp.mod_inverse(3,26) == mod_inverse(3,26)

### 1.4 Matrix Operations

Building upon the fundamental matrix operations of addition, multiplication 
and linear transformations, we introduce three additional properties 
required for the Hill cipher.

**Determinant**

The determinant is a scalar value that captures whether a matrix is 
invertible. For a $2 \times 2$ matrix:

$$K = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$

it is computed as:

$$\det(K) = ad - bc$$

This is a special case of the general $n \times n$ cofactor expansion 
along row $i$:

$$\det(A) = \sum_{j=1}^{n} (-1)^{i+j} \, a_{ij} \, M_{ij}$$

where $M_{ij}$ is the determinant of the submatrix formed by deleting 
row $i$ and column $j$, and $(-1)^{i+j}$ is the sign of the cofactor. 
Throughout this project all key matrices are $2 \times 2$, so the 
formula $ad - bc$ is sufficient.

Geometrically, the determinant represents the signed area of the 
parallelogram formed by the two column vectors of $K$. When 
$\det(K) = 0$, the two columns are linearly dependent — the matrix 
is singular and has no inverse.

Over $\mathbb{R}$ this is the only condition for non-invertibility. 
Over $\mathbb{Z}_{26}$ the condition is stricter — the determinant 
must additionally satisfy:

$$\gcd(\det(K),\ 26) = 1$$

For example, given:

$$K = \begin{pmatrix} 3 & 3 \\ 2 & 5 \end{pmatrix}$$

$$\det(K) = 3 \cdot 5 - 3 \cdot 2 = 9, \qquad \gcd(9, 26) = 1 \checkmark$$

In [9]:
def determinant_for_2n_matrix(matrix) -> int:
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

In [10]:
matrix = np.array([[3, 3],
              [2, 5]])
print(determinant_for_2n_matrix(matrix))
print(sp.gcd(determinant_for_2n_matrix(matrix),26))

9
1


SymPy provides a built-in det function which we use to verify our implementation:

In [11]:
assert determinant_for_2n_matrix(matrix) == sp.Matrix(matrix).det()

**Adjugate**

The adjugate is the matrix that satisfies:

$$K \cdot \text{adj}(K) = \det(K) \cdot I$$

Dividing both sides by $\det(K)$ yields the inverse — this is the 
algebraic purpose of the adjugate. It is defined as the transpose 
of the cofactor matrix. For an $n \times n$ matrix, each entry is:

$$\text{adj}(A)_{ij} = (-1)^{i+j} \, M_{ji}$$

where $M_{ji}$ is the determinant of the submatrix formed by 
deleting row $j$ and column $i$, and $(-1)^{i+j}$ is the 
alternating sign pattern:

$$\begin{pmatrix} + & - & + & \cdots \\ - & + & - & \cdots \\ 
+ & - & + & \cdots \\ \vdots & & & \ddots \end{pmatrix}$$

For a $2 \times 2$ matrix this reduces to a direct formula — 
swap the diagonal elements and negate the off-diagonal elements:

$$\text{adj}\begin{pmatrix} a & b \\ c & d \end{pmatrix} = 
\begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$$

The fundamental algebraic property of the adjugate is:

$$K \cdot \text{adj}(K) = \det(K) \cdot I$$

Applying this to our example:

$$\text{adj}(K) = \begin{pmatrix} 5 & -3 \\ -2 & 3 \end{pmatrix}$$

In [12]:
def adjugate_for_2n_matrix(matrix) -> np.array:
    return np.array([[matrix[1][1] , -matrix[0][1]],
                     [-matrix[1][0],matrix[0][0]]])

In [13]:
adj = adjugate_for_2n_matrix(matrix)

print(adj)
assert np.array_equal(matrix @ adj, sp.Matrix(matrix).det() * np.eye(2, dtype=int))

[[ 5 -3]
 [-2  3]]



**Modular Matrix Inverse**

Over $\mathbb{R}$ the matrix inverse is given by:

$$K^{-1} = \frac{1}{\det(K)} \cdot \text{adj}(K)$$

Over $\mathbb{Z}_{26}$, scalar division does not exist — it is 
replaced by the modular inverse of the determinant, established 
in Section 1.3:

$$K^{-1} \equiv \det(K)^{-1} \cdot \text{adj}(K) \pmod{26}$$

This formula is valid if and only if $\gcd(\det(K),\ 26) = 1$. 
The computation proceeds in three steps.

**Step 1** — verify the invertibility condition:

$$\det(K) = ad - bc, \qquad \gcd(\det(K),\ 26) = 1$$

**Step 2** — find the modular inverse of the determinant:

$$\det(K)^{-1} \equiv x \pmod{26} \quad \text{where} \quad 
x \cdot \det(K) \equiv 1 \pmod{26}$$

**Step 3** — multiply and reduce every entry modulo 26:

$$K^{-1} \equiv \det(K)^{-1} \cdot \begin{pmatrix} d & -b \\ -c & a 
\end{pmatrix} \pmod{26}$$

Note that negative entries arising from the adjugate are reduced 
into $\mathbb{Z}_{26}$ by adding 26:

$$-9 \bmod 26 = 17, \qquad -6 \bmod 26 = 20$$

Applying all three steps to our example:

$$9^{-1} \equiv 3 \pmod{26} \quad \text{since} \quad 
9 \times 3 = 27 \equiv 1 \pmod{26}$$

$$K^{-1} \equiv 3 \cdot \begin{pmatrix} 5 & -3 \\ -2 & 3 \end{pmatrix} 
\equiv \begin{pmatrix} 15 & -9 \\ -6 & 9 \end{pmatrix} 
\equiv \begin{pmatrix} 15 & 17 \\ 20 & 9 \end{pmatrix} \pmod{26}$$

The result satisfies the identity:

$$K \cdot K^{-1} \equiv \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} 
\pmod{26}$$

A $2 \times 2$ integer matrix $K$ is a valid Hill cipher key 
**if and only if**:

$$\gcd(\det(K),\ 26) = 1$$

This ensures $K^{-1} \pmod{26}$ exists and that decryption is 
uniquely defined.

The following implements these three operations step by step, 
then verifies the identity $K \cdot K^{-1} \equiv I \pmod{26}$ 
both manually and using SymPy:

In [69]:
def matrix_modular_inverse(matrix):
    det = sp.Matrix(matrix).det()
    det_mod_inverse = mod_inverse(det,26)
    adj = adjugate_for_2n_matrix(matrix)
    result = (det_mod_inverse * adj) % 26
    return result
matrix_inversed = matrix_modular_inverse(matrix)
sympy_inversed = sp.Matrix(matrix).inv_mod(26)

assert np.array_equal(matrix_inversed ,sympy_inversed)
assert np.array_equal((matrix_inversed @ matrix) % 26,  np.eye(2,dtype = int))
assert np.array_equal((sympy_inversed @ matrix) % 26,  np.eye(2,dtype = int))

## 2. Classical Ciphers

Before the advent of modern cryptography, secret communication relied on
**classical ciphers** — schemes that operate on individual letters or small
groups of letters using simple mathematical rules. While no longer secure
by modern standards, these ciphers introduce the core ideas that underpin
all subsequent cryptographic systems: key-based transformation, modular
arithmetic over an alphabet, and the requirement for an invertible
operation.

In this section we examine two classical substitution ciphers and analyze
their vulnerability to statistical cryptanalysis:

- **Caesar cipher** — a shift cipher in which every letter is displaced
  by a fixed constant $k$ modulo 26, representing the simplest possible
  application of modular arithmetic to a message.
- **Vigenère cipher** — a polyalphabetic extension of the Caesar cipher
  in which the shift varies with each letter according to a repeating
  keyword, significantly increasing resistance to frequency analysis.
- **Frequency analysis** — a ciphertext-only attack that exploits the
  fact that natural language has a predictable letter distribution.
  Applied to both ciphers above, it demonstrates why monoalphabetic
  substitution offers no practical security and why polyalphabetic
  schemes are substantially harder to break.

Both ciphers operate over $\mathbb{Z}_{26}$, the same modular space
established in Section 1.1, and share the same encryption/decryption
structure — a modular addition and its inverse. The mathematical
simplicity of these schemes makes their weaknesses equally transparent,
setting the stage for the more robust matrix-based Hill cipher in
Section 3.

### 2.1 Caesar Cipher

The Caesar cipher is one of the oldest known encryption schemes, attributed
to Julius Caesar who used it to protect military correspondence. It operates
by shifting every letter of the plaintext by a fixed constant $k$, known as
the **key**, within the 26-letter alphabet.

**Encryption**

Each plaintext letter is first mapped to its numerical equivalent in
$\mathbb{Z}_{26}$, where $\texttt{A} = 0, \texttt{B} = 1, \dots,
\texttt{Z} = 25$. The ciphertext letter $c$ is then obtained by:

$$c \equiv p + k \pmod{26}$$

where $p \in \mathbb{Z}_{26}$ is the numerical value of the plaintext
letter and $k \in \mathbb{Z}_{26}$ is the shift key.

**Decryption**

Decryption reverses the shift by subtracting the key:

$$p \equiv c - k \pmod{26}$$

Since subtraction modulo 26 is always defined, every key $k$ is valid —
the Caesar cipher does not require the coprimality condition established
in Section 1.2. This is because the operation is addition, not
multiplication.

**Example**

Encrypting the message $\texttt{MATH}$ with key $k = 3$:

$$\texttt{M} = 12 \rightarrow (12 + 3) \bmod 26 = 15 = \texttt{P}$$
$$\texttt{A} = 0  \rightarrow (0  + 3) \bmod 26 = 3  = \texttt{D}$$
$$\texttt{T} = 19 \rightarrow (19 + 3) \bmod 26 = 22 = \texttt{W}$$
$$\texttt{H} = 7  \rightarrow (7  + 3) \bmod 26 = 10 = \texttt{K}$$

$$\texttt{MATH} \xrightarrow{k=3} \texttt{PDWK}$$

**Security**

The Caesar cipher has a keyspace of only 25 non-trivial shifts, making
it trivially vulnerable to brute force. More fundamentally, it is a
**monoalphabetic substitution cipher** — each plaintext letter always
maps to the same ciphertext letter, preserving the natural letter
frequency distribution of the underlying language. This property is
formally exploited in Section 2.3.

The following cells implement encryption and decryption:

In [28]:
def caesar_cipher_encryption(p , k):
    chars = (np.frombuffer(p.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    chars = np.where((chars >= 0) & (chars <= 25),(chars + k) % 26 , chars )
    return ''.join((chars + ord('A')).astype(np.uint8).tobytes().decode())

In [29]:
plaintext = "MATH"
encrypted = caesar_cipher_encryption(plaintext,3)
print(encrypted)

PDWK


The following cell implements decryption:

In [30]:
def caesar_cipher_decryption(c , k) :
    chars = (np.frombuffer(c.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    chars = np.where((chars >= 0) & (chars <= 25),(chars - k) % 26 , chars )
    return ''.join((chars + ord('A')).astype(np.uint8).tobytes().decode())

In [31]:
decrypted = caesar_cipher_decryption(encrypted,3)
print(decrypted)
assert decrypted == plaintext

MATH


### 2.2 Vigenère Cipher

The Vigenère cipher is a polyalphabetic substitution cipher that extends
the Caesar cipher by using a **keyword** rather than a single shift. Each
letter of the plaintext is shifted by the corresponding letter of the
keyword, with the keyword repeating cyclically for messages longer than
the key.

**Encryption**

Let the plaintext be $p_0, p_1, \dots, p_{n-1}$ and the keyword have
length $m$, with numerical values $k_0, k_1, \dots, k_{m-1} \in
\mathbb{Z}_{26}$. The ciphertext letter at position $i$ is:

$$c_i \equiv p_i + k_{i \bmod m} \pmod{26}$$

The index $i \bmod m$ ensures the keyword repeats cyclically across the
full length of the message.

**Decryption**

Decryption subtracts the corresponding keyword letter at each position:

$$p_i \equiv c_i - k_{i \bmod m} \pmod{26}$$

**Example**

Encrypting $\texttt{MATHISCOOL}$ with keyword $\texttt{KEY}$
$(k = [10, 4, 24])$:

$$\begin{array}{ccccccccccc}
\text{Plaintext:}  & \texttt{M} & \texttt{A} & \texttt{T} & \texttt{H} & \texttt{I} & \texttt{S} & \texttt{C} & \texttt{O} & \texttt{O} & \texttt{L} \\
\text{Key:}        & \texttt{K} & \texttt{E} & \texttt{Y} & \texttt{K} & \texttt{E} & \texttt{Y} & \texttt{K} & \texttt{E} & \texttt{Y} & \texttt{K} \\
\text{Ciphertext:} & \texttt{W} & \texttt{E} & \texttt{R} & \texttt{R} & \texttt{M} & \texttt{Q} & \texttt{M} & \texttt{S} & \texttt{M} & \texttt{V}
\end{array}$$

**Security**

Unlike the Caesar cipher, the Vigenère cipher is **polyalphabetic** —
the same plaintext letter maps to different ciphertext letters depending
on its position, which disrupts the direct letter frequency correspondence
exploited in Section 2.3. The effective keyspace grows exponentially with
keyword length, making brute force infeasible for long keywords. However,
once the keyword length is known — recoverable via the Kasiski test or
index of coincidence — the cipher reduces to $m$ independent Caesar
ciphers, each breakable by frequency analysis.

The following cells implement encryption and decryption:

In [32]:
def vigenere_cipher_encryption(p,k):
    chars = (np.frombuffer(p.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    key = (np.frombuffer(k.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    key = np.tile(key, len(chars) // len(key) + 1)[:len(chars)]

    chars = np.where((chars >= 0  ) & (chars <= 25) , (chars + key ) % 26 , chars)
    return ''.join((chars + ord('A')).astype(np.uint8).tobytes().decode())
    

In [33]:
plaintext = "MATHISCOOL"
key = "KEY"
example_encrypted_text = "WERRMQMSMV"
encrypted = vigenere_cipher_encryption(plaintext,key)
print(encrypted)
assert encrypted == example_encrypted_text

WERRMQMSMV


The following cell implements decryption:

In [34]:
def vigenere_cipher_decryption(c , k): 
    chars = (np.frombuffer(c.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    key = (np.frombuffer(k.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    key = np.tile(key, len(chars) // len(key) + 1)[:len(chars)]

    chars = np.where((chars >= 0  ) & (chars <= 25) , (chars - key ) % 26 , chars)
    return ''.join((chars + ord('A')).astype(np.uint8).tobytes().decode())

In [35]:
decrypted = vigenere_cipher_decryption(encrypted , key) 
print(decrypted)
assert plaintext == decrypted

MATHISCOOL


**Attacking the Vigenère Cipher**

The Vigenère cipher is polyalphabetic, so a direct frequency comparison
fails — the same plaintext letter maps to different ciphertext letters
at different positions, flattening the frequency distribution. The attack
proceeds in two stages.

**Stage 1 — Recover the keyword length** using two complementary methods:
the **Kasiski Test** and the **Index of Coincidence** (IC).

The **Kasiski Test** exploits the fact that when the same plaintext segment
aligns with the same portion of the repeating key, it produces an identical
ciphertext trigram. The distance between any two such repetitions is
necessarily a multiple of the keyword length. Formally, if trigram $T$
appears at positions $p_1$ and $p_2$, then:

$$m \mid (p_2 - p_1)$$

The keyword length is estimated as the value that divides the greatest
number of observed distances. However, not all repeated trigrams arise
from key alignment — some occur by coincidence, introducing distances
unrelated to the keyword length. These spurious distances corrupt a naive
$\gcd$ computation. The robust approach is therefore to count, for each
candidate length $m$, how many distances it divides evenly, and select
the $m$ with the most support:

$$m^* = \operatorname{argmax}_{m \geq 2} \left| \{ d \in D : m \mid d \} \right|$$

where $D$ is the multiset of all observed inter-trigram distances.

The **Index of Coincidence** (IC) serves as a fallback when the ciphertext
is too short to produce reliable trigram repetitions. It is defined as the
probability that two randomly selected ciphertext letters are identical:

$$\text{IC}(m) = \frac{1}{m} \sum_{j=0}^{m-1} \frac{\displaystyle
\sum_{i=0}^{25} f_{i,j}(f_{i,j} - 1)}{N_j(N_j - 1)}$$

where $f_{i,j}$ is the frequency of letter $i$ in sub-sequence $j$,
and $N_j$ is the length of sub-sequence $j$. For English plaintext
$\text{IC} \approx 0.065$, while for a uniformly random text
$\text{IC} \approx 0.038$. When the keyword length $m$ is correct, each
sub-sequence behaves like a Caesar cipher over English text, driving the
average IC toward $0.065$. The combined detection strategy is therefore:

$$m^* = \begin{cases} \text{Kasiski}(C) & \text{if sufficient trigram
repetitions exist} \\ \operatorname{argmax}_{2 \leq m \leq \lfloor N/3
\rfloor} \ \text{IC}(m) & \text{otherwise} \end{cases}$$

A fundamental limitation of both methods is that they require sufficient
ciphertext length. The Kasiski test requires enough text for meaningful
trigram repetitions to occur — coincidental repetitions introduce noise
that can corrupt the distance analysis. The IC method requires each
sub-sequence to contain enough characters for the observed frequency
distribution to converge to the true English distribution. For a keyword
of length $m$, each sub-sequence contains approximately $N/m$ characters.
When $N/m$ is small, the IC signal becomes unreliable and both methods
may return an incorrect keyword length.

**Stage 2 — Break each Caesar sub-cipher** independently. Once the
keyword length $m$ is known, the ciphertext is split into $m$
sub-sequences — one for each key position:

$$C_j = \{ c_{j}, c_{j+m}, c_{j+2m}, \dots \} \quad j = 0, 1, \dots, m-1$$

Each sub-sequence $C_j$ is an independent Caesar cipher with shift
$k_j$. Applying the dot product frequency attack from Section 2.3
to each sub-sequence recovers the individual shifts, and the full
keyword is reconstructed as:

$$\texttt{KEY} = (k_0,\ k_1,\ \dots,\ k_{m-1})$$

To demonstrate, we encrypt with keyword $\texttt{KEY}$ $(m = 3,\
k = [10, 4, 24])$:

$$\texttt{MATHEMATICSISTHEFOUNDATION} \xrightarrow{\texttt{KEY}}
\texttt{WOMLDROBOMSSMXLIDYSCRHOFKSW}$$

The Kasiski test identifies repeated trigrams with distances divisible
by $3$. The IC peaks at $m = 3$, confirming the keyword length.
Applying the Caesar attack to each sub-sequence recovers
$k_0 = 10,\ k_1 = 4,\ k_2 = 24$, corresponding exactly to
$\texttt{K}$, $\texttt{E}$, $\texttt{Y}$.

The following cells implement both attacks and visualise the letter
frequency distributions using Matplotlib:

In [475]:
english_freq = np.array([8.2, 1.5, 2.8, 4.3, 12.7, 2.2, 2.0, 6.1, 7.0, 0.2,
                         0.8, 4.0, 2.4, 6.7, 7.5, 1.9, 0.1, 6.0, 6.3, 9.1,
                         2.8, 1.0, 2.4, 0.2, 2.0, 0.1])

def attacking_caesar(cipher_text):
    
    chars = (np.frombuffer(cipher_text.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    chars = chars[(chars >= 0) & (chars <= 25)]
    ordered_chars = np.bincount(chars , minlength= 26)
    shifts = np.array([np.roll(ordered_chars, k) for k in range(26)])
    scores = shifts @ english_freq
    most_frequent = np.argmax(scores)
    return caesar_cipher_decryption((cipher_text) ,(26 - most_frequent) % 26)


In [476]:

text = "MATHEMATICSISTHEFOUNDATIONOFCRYPTOGRAPHY"
encrypted = caesar_cipher_encryption(text , 3)
result =  attacking_caesar(encrypted)
print(result)
assert text == result

MATHEMATICSISTHEFOUNDATIONOFCRYPTOGRAPHY


In [477]:
def findIC(subset):
    subset = subset[subset <= 25]
    letters = np.bincount(subset, minlength=26)
    return np.sum(letters * (letters - 1))/(len(subset) * (len(subset) - 1))

In [478]:
def score_english(result):
    result = result % 26
    observed = np.bincount(result, minlength=26)
    expected = english_freq * len(result)
    chi2 = np.sum((observed - expected) ** 2 / (expected + 1e-9))
    return -chi2

In [479]:
def decrypt_caesar_array(subset): 
    ordered_chars = np.bincount(subset, minlength=26)
    shifts = np.array([np.roll(ordered_chars, k) for k in range(26)])
    scores = shifts @ english_freq
    key = (26 - np.argmax(scores)) % 26
    
    return (subset - key) % 26

In [480]:
def break_vigenere(cipher_text):
    chars = (np.frombuffer(cipher_text.upper().encode(), dtype=np.uint8) - ord('A')).astype(int)
    max_length = min(20, len(chars) // 3)
    avg_Ics = np.zeros(max_length)
    for m in range(1, max_length + 1):
        subsets = [chars[i::m] for i in range(m)]
        avg_Ics[m - 1] = np.mean([findIC(s) for s in subsets])

    english_ic_threshold = 0.055
    candidates = [i + 1 for i, ic in enumerate(avg_Ics) if ic > english_ic_threshold]
    minimum = min(candidates) if candidates else np.argmax(avg_Ics) + 1

    subsets = [chars[i::minimum] for i in range(minimum)]
    decrypted_subsets = [decrypt_caesar_array(s) for s in subsets]
    result = np.empty(len(chars), dtype=int)
    for i, decrypted_subset in enumerate(decrypted_subsets):
        result[i::minimum] = decrypted_subset

    return ''.join((result + ord('A')).astype(np.uint8).tobytes().decode())

In [481]:
key = "KEY"
vigenere_demo = (
    "MATHEMATICSISTHEFOUNDATIONOFCRYPTOGRAPHYANDITSAPPLICATIONSARE"
    "EVERYWHEREINMODERNLIFEFROMBANKINGTOINTERNETCOMMUNICATIONSTHE"
    "STUDYOFPRIMESANDMODULARARITHMETICUNDERPINSEVERYSECUREPROTOCOL"
    "WEUSETODAYFROMSSLCERTIFICATESTODIGITALSIGNATURESANDBLOCKCHAIN"
    "TECHNOLOGYWITHOUTMATHEMATICSMODERNENCRYPTIONWOULDBEIMPOSSIBLE"
)
encrypt_vigenere = vigenere_cipher_encryption(vigenere_demo,key)

dec = break_vigenere(encrypt_vigenere)
print(enc)
assert vigenere_demo == dec

MATHEMATICSISTHEFOUNDATIONOFCRYPTOGRAPHYANDITSAPPLICATIONSAREEVERYWHEREINMODERNLIFEFROMBANKINGTOINTERNETCOMMUNICATIONSTHESTUDYOFPRIMESANDMODULARARITHMETICUNDERPINSEVERYSECUREPROTOCOLWEUSETODAYFROMSSLCERTIFICATESTODIGITALSIGNATURESANDBLOCKCHAINTECHNOLOGYWITHOUTMATHEMATICSMODERNENCRYPTIONWOULDBEIMPOSSIBLE
